# Deutsch-Jozsa Quantum Algorithm Analysis

In [3]:
import random
from itertools import product
from typing import Callable

## Problem 1: Generating Random Boolean Functions.

In the Deutsch-Jozsa setting, a **constant** Boolean function returns the same value for every input, while a **balanced** Boolean function returns True for exactly half of the inputs and False for the other half. These two categories are the core promise the algorithm exploits.

**Research:** A 4-bit input space has $2^4 = 16$ possible inputs. A balanced 4-bit function therefore returns True for exactly 8 inputs and False for the remaining 8. This distinction is crucial for validating that the Deutsch-Jozsa algorithm can separate constant from balanced functions with a single query (see the overview in [IBM Quantum Learning](https://learning.quantum.ibm.com/course/fundamentals-of-quantum-algorithms/quantum-oracles)).

> Programming perspective: to guarantee balance, we construct a list with eight True values and eight False values, then shuffle it to randomize the mapping while preserving the 8/8 split.

In [4]:
def random_constant_balanced(seed: int | None = None) -> Callable[[bool, bool, bool, bool], bool]:
    """Return a random 4-bit Boolean function that is constant or balanced.

    Args:
        seed: Optional seed for reproducibility.

    Returns:
        A callable f(a, b, c, d) -> bool representing the Boolean function.
    """
    # Create a local RNG to avoid global state and enable reproducibility.
    rng = random.Random(seed)

    # Randomly pick whether the function is constant or balanced.
    function_type = rng.choice(["constant", "balanced"])

    # Enumerate all 16 possible 4-bit inputs.
    inputs = list(product([False, True], repeat=4))

    if function_type == "constant":
        # Choose the single value returned for every input.
        constant_value = rng.choice([False, True])

        def oracle(a: bool, b: bool, c: bool, d: bool) -> bool:
            """Constant 4-bit Boolean function."""
            # Ignore inputs to return the same value every time.
            return constant_value

        return oracle

    # For a balanced function, assign exactly 8 True and 8 False values.
    outputs = [True] * 8 + [False] * 8
    # Shuffle to randomize the mapping while preserving balance.
    rng.shuffle(outputs)

    mapping = dict(zip(inputs, outputs))

    def oracle(a: bool, b: bool, c: bool, d: bool) -> bool:
        """Balanced 4-bit Boolean function."""
        # Look up the output for the given input tuple.
        return mapping[(a, b, c, d)]

    return oracle

In [5]:
from itertools import product

# Reproducible example using a fixed seed.
test_oracle = random_constant_balanced(seed=42)

truth_table = {
    inputs: test_oracle(*inputs)
    for inputs in product([False, True], repeat=4)
}

true_count = sum(truth_table.values())
false_count = len(truth_table) - true_count

if true_count in (0, 16):
    function_label = "constant"
elif true_count == 8:
    function_label = "balanced"
else:
    function_label = "unexpected"

print("Function type:", function_label)
print("True count:", true_count, "False count:", false_count)
print("Sample outputs (first 5):", list(truth_table.items())[:5])

Function type: constant
True count: 0 False count: 16
Sample outputs (first 5): [((False, False, False, False), False), ((False, False, False, True), False), ((False, False, True, False), False), ((False, False, True, True), False), ((False, True, False, False), False)]


## Problem 2: Classical Testing for Function Type

In the classical setting, we may need to evaluate many inputs to decide whether a promised Boolean function is constant or balanced. This makes classical testing a baseline for comparison with the Deutsch-Jozsa quantum approach, which can identify the type with far fewer oracle queries under the same promise.

For an $n$-bit function, there are $2^n$ possible inputs. In the worst classical case, you must inspect one more than half of the truth table to rule out the opposite class, so query complexity scales as $O(2^n)$. For 4 bits, that means at most $2^{4-1}+1 = 9$ queries.

Reference: https://en.wikipedia.org/wiki/Deutsch%E2%80%93Jozsa_algorithm

In [1]:
from itertools import product
from typing import Callable


def determine_constant_balanced(
    f: Callable[[bool, bool, bool, bool], bool],
) -> str:
    """Classify a promised 4-bit Boolean function as constant or balanced.

    This classical routine evaluates up to 9 inputs in the worst case
    ($2^{n-1} + 1$ for $n=4$), which is enough to distinguish constant from
    balanced under the Deutsch-Jozsa promise.

    Args:
        f: A Boolean function taking four Boolean inputs (a, b, c, d) and
            returning a Boolean output.

    Returns:
        "constant" if all queried outputs match, otherwise "balanced" when
        two different output values are observed.
    """
    input_combinations = list(product([False, True], repeat=4))

    first_result = f(*input_combinations[0])
    query_count = 1

    # For n=4, classical worst-case checks are capped at 2^(n-1)+1 = 9 calls.
    for combination in input_combinations[1:]:
        if query_count >= 9:
            break

        current_result = f(*combination)
        query_count += 1

        # Early exit: differing outputs immediately prove the function balanced.
        if current_result != first_result:
            return "balanced"

    return "constant"

In [2]:
def constant_true(a: bool, b: bool, c: bool, d: bool) -> bool:
    return True


def constant_false(a: bool, b: bool, c: bool, d: bool) -> bool:
    return False


def balanced_example(a: bool, b: bool, c: bool, d: bool) -> bool:
    return a ^ b ^ c


balanced_true_count = sum(
    balanced_example(a, b, c, d)
    for a, b, c, d in product([False, True], repeat=4)
)

print("Balanced example true-count:", balanced_true_count, "of 16")
print("constant_true   ->", determine_constant_balanced(constant_true))
print("constant_false  ->", determine_constant_balanced(constant_false))
print("balanced_example->", determine_constant_balanced(balanced_example))

Balanced example true-count: 8 of 16
constant_true   -> constant
constant_false  -> constant
balanced_example-> balanced


**Efficiency note:** This classical baseline needs up to $2^{n-1}+1$ oracle calls in the promised Deutsch-Jozsa setting, so runtime grows exponentially with input size ($O(2^n)$). By contrast, the quantum Deutsch-Jozsa approach is designed to separate constant and balanced functions with dramatically fewer queries; see [IBM Quantum Learning](https://learning.quantum.ibm.com/).

## Problem 3: Quantum Oracles.

## Problem 4: Deutsch's Algorithm with Qiskit.

## Problem 5: Scaling to the Deutsch–Jozsa Algorithm.